# Car detector: cloud training
Runtime > Change runtime type > GPU (T4). Upload `car-data.zip` with the Files panel, then run all cells. Each run is copied to Google Drive (`MyDrive/car-runs`) as soon as it finishes, so a disconnect loses at most the run in progress.

In [ ]:
!pip -q install ultralytics
!unzip -qo car-data.zip -d /content
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import time, shutil, pathlib
from ultralytics import YOLO
BASE = '/content/yolocar-11n.pt'
SAVE = pathlib.Path('/content/drive/MyDrive/car-runs'); SAVE.mkdir(exist_ok=True)
COMMON = dict(data='/content/yolo/data.yaml', imgsz=1536, device=0, project='/content/runs', exist_ok=True,
              optimizer='AdamW', flipud=0.5, fliplr=0.5, degrees=0, patience=0, workers=2, plots=False, verbose=False)
# name: (weights to start from, overrides). Earlier: 60 epochs 0.808 mAP50, 120 epochs 0.821.
EXPERIMENTS = {
  'n_120':      (BASE, dict(epochs=120, lr0=0.0005, batch=8)),
  'n_lr1e3':    (BASE, dict(epochs=120, lr0=0.001, batch=8)),
  'n_seed1':    (BASE, dict(epochs=120, lr0=0.0005, batch=8, seed=1)),
  'n_img1920':  (BASE, dict(epochs=120, lr0=0.0005, batch=4, imgsz=1920)),
  's_lr5e4':    ('yolo11s.pt', dict(epochs=120, lr0=0.0005, batch=4)),
}
for name, (start, over) in EXPERIMENTS.items():
    if (SAVE / name / 'weights' / 'best.pt').exists():
        print(name, 'already saved, skipping'); continue
    t = time.time()
    YOLO(start).train(name=name, **{**COMMON, **over})
    shutil.copytree(f'/content/runs/{name}', SAVE / name, dirs_exist_ok=True)
    print(name, 'done in', round((time.time()-t)/60, 1), 'min, copied to Drive')

In [ ]:
import pandas as pd
for name in EXPERIMENTS:
    f = SAVE / name / 'results.csv'
    if f.exists():
        df = pd.read_csv(f); df.columns = df.columns.str.strip()
        print(name, df[['metrics/precision(B)','metrics/recall(B)','metrics/mAP50(B)','metrics/mAP50-95(B)']].iloc[-1].round(3).to_dict())